# Hot swap the live model in the Serve app

This notebook changes which model answers live predictions in a running Anyscale Service, with no
restart and no new cluster. It sends one molecule before and after the swap, and compares each pose
with the molecule's crystal structure.

**The demo:** EV-A71 2A compound x7259a, a potent late follow-on (pKD 7.9). The fine-tuned model never
trained on it.

| Pose from | Benchmark RMSD to the crystal | Benchmark success on follow-ons |
|---|---|---|
| Boltz-2, a general model (before) | 13.2 Å, wrong but confident (ipTM 0.84) | 1 to 5% |
| OpenFold3 fine-tuned on the fragment screen (after) | 0.4 Å, correct | 78 to 79% |

**How the swap works**

- The `Viewer` deployment takes `{"pose_from": "Boltz2Live" | "FineTunedLive"}` as its Ray Serve
  `user_config`. The model deployments take `{"version", "checkpoint", "name"}`.
- `anyscale.service.deploy(..., in_place=True)` with a config that changes only `user_config` calls
  `reconfigure()` on the running replicas. Ray Serve does not restart them.

**Before you start**

- Run this notebook in a Workspace, from the repository folder. It deploys `service.yaml` from here.
- The data must be in `/mnt/shared_storage/aiddd`.
- Do not change files in this folder between the first deploy and the swap. A code change makes the
  update restart every replica. Notebooks are excluded from the upload, so editing this one is safe.

In [ ]:
import json, os, time
from pathlib import Path

import anyscale
import pandas as pd
import requests
from anyscale.service.models import ServiceConfig

SHARED = Path(os.environ.get("AIDDD_SHARED", "/mnt/shared_storage/aiddd"))
assert (SHARED / "results" / "ev71-2a").exists(), f"missing {SHARED}/results: copy the data first"

# Your own Service. "aiddd-viewer" is the shared workshop Service: never use it for practice.
SERVICE_NAME = "aiddd-viewer-geoff"
assert SERVICE_NAME != "aiddd-viewer", "pick your own Service name for practice runs"

# x7259a: Boltz-2 is wrong, the fine-tuned OpenFold3 is right. Backups: x7570a, x7056a.
SMILES = "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"

# user_config per deployment, before and after the swap.
BEFORE = {"Viewer": {"pose_from": "Boltz2Live"}}
AFTER = {"Viewer": {"pose_from": "FineTunedLive"}}

Other swaps use the same cells. For example, a new OpenFold3 checkpoint from `finetune_ray_train.py`:

```python
AFTER = {"FineTunedLive": {"version": "ray-ft-1", "name": "OF3 Ray Train fine-tune",
                           "checkpoint": str(SHARED / "finetune/runs/<run>/checkpoint_<n>/of3_ev71_ft.pt")}}
```

## Helpers

`service_config()` is `service.yaml` with your Service name and a `user_config` for each listed
deployment. `predict()` calls the Viewer's `/predict` endpoint, the same one the web page uses.

In [ ]:
def service_config(user_configs: dict) -> ServiceConfig:
    """service.yaml, renamed, with {deployment name: user_config} overrides."""
    cfg = ServiceConfig.from_yaml("service.yaml", name=SERVICE_NAME)
    deployments = [{"name": d, "user_config": uc} for d, uc in user_configs.items()]
    return cfg.options(applications=[{**app, "deployments": deployments} for app in cfg.applications])


def predict(smiles: str) -> dict:
    s = anyscale.service.status(name=SERVICE_NAME)
    headers = {"Authorization": f"Bearer {s.query_auth_token}"} if s.query_auth_token else {}
    r = requests.post(s.query_url.rstrip("/") + "/predict", json={"smiles": smiles},
                      headers=headers, timeout=600)
    return r.json()


def summary(out: dict) -> dict:
    """The model behind each part of one /predict response, and the main numbers."""
    models, key, m = out.get("models") or {}, out.get("answer_key") or {}, out.get("metrics", {})
    return {"pose model": f"{models.get('pose', {}).get('name')} ({models.get('pose', {}).get('version')})",
            "affinity model": f"{models.get('affinity', {}).get('name')} ({models.get('affinity', {}).get('version')})",
            "crystal": key.get("complex"), "ligand RMSD (Å)": key.get("lig_rmsd"),
            "pose ipTM": out.get("pose_iptm"), "affinity (log10 IC50, µM)": m.get("affinity_pred_value"),
            "status": out.get("status"), "seconds": out.get("seconds")}

## 1. Start the Service in the "before" state

The first deploy starts a new cluster with two L4 nodes, which takes about 5 to 10 minutes. If the
Service is already running, this cell only resets it to `BEFORE`, in place.

In [ ]:
try:
    running = str(anyscale.service.status(name=SERVICE_NAME).state) == "RUNNING"
except Exception:   # no Service with this name yet
    running = False
anyscale.service.deploy(service_config(BEFORE), in_place=running, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=1800)
print(anyscale.service.status(name=SERVICE_NAME).query_url)

## 2. Before the swap

A new molecule takes one to three minutes. Repeat requests return from the cache. If this still shows
the fine-tuned model, the reset in step 1 has not reached the replicas yet: run the cell again.

In [ ]:
before = predict(SMILES)
summary(before)

## 3. Hot swap

`in_place=True` reuses the running cluster. Only `user_config` changes, so Ray Serve calls
`reconfigure()` on the live replicas. The loop sends the same molecule until the response comes from
the new configuration, which is usually the first try.

In [ ]:
t0 = time.time()
anyscale.service.deploy(service_config(AFTER), in_place=True, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=600)
for _ in range(30):
    after = predict(SMILES)
    if after.get("models") != before.get("models"):
        break
    time.sleep(5)
print(f"swap took {time.time() - t0:.0f} s, including the first prediction after it")
summary(after)

## 4. Before and after

In [ ]:
pd.DataFrame({"before": summary(before), "after": summary(after)})

The full response metadata, as the web page receives it:

In [ ]:
print(json.dumps({k: after.get(k) for k in ("models", "answer_key", "pose_model", "pose_iptm")}, indent=2))

## 5. Swap back

The same in-place update, with the `BEFORE` configuration.

In [ ]:
anyscale.service.deploy(service_config(BEFORE), in_place=True, name=SERVICE_NAME)

## 6. Stop the Service

The Service keeps two L4 nodes running until you stop it.

In [ ]:
# anyscale.service.terminate(name=SERVICE_NAME)